import mysql.connector                 
from mysql.connector import errorcode

NO EJECUTAR

try:
    # nos conectamos al servidor de mysql, no especificamos schema porque vamos a crearlo
    
    cnx = mysql.connector.connect(
        user='root',
        password= input('password_MySQL'),
        host='127.0.0.1'
        # auth_plugin = 'mysql_native_password'
    )
    print("¡Conexión exitosa a MySQL!") #
except mysql.connector.Error as err:
    # Si es un error de acceso denegado (ej. contraseña o usuario incorrecto)
    if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
        print("Algo está mal con tu nombre de usuario o contraseña.") #
    # Si la base de datos no existe
    elif err.errno == errorcode.ER_BAD_DB_ERROR:
        print("La base de datos no existe.") #
    # Para cualquier otro tipo de error
    else:
        print(err) #
        print("Código de Error:", err.errno) #
        print("SQLSTATE", err.sqlstate) #
        print("Mensaje", err.msg) #

In [8]:
import mysql.connector   # Importamos las librarias necesarias
import pandas as pd
from mysql.connector import Error

class BaseDatosHR:
    """
    Clase para crear e insertar datos en una Base de Datos relacional MySQL
    de Recursos Humanos - ABC Corporation
    """
    
    def __init__(self, host='127.0.0.1', user='root', password=input('password_MySQL'), database='abc_hr', auth_plugin = 'mysql_native_password'):
        self.host = host
        self.user = user
        self.password = password
        self.database = database
        self.conexion = None
        self.cursor = None
    
    def conectar_sin_bd(self):
        """Establece conexión con MySQL sin especificar base de datos"""
        try:
            conexion = mysql.connector.connect(
                host=self.host,
                user=self.user,
                password=self.password,
                auth_plugin = 'mysql_native_password'
            )
            return conexion
        except Error as e:
            print(f"✗ Error de conexión: {e}")
            return None
    
    def crear_base_de_datos(self):
        """Crea la base de datos si no existe"""
        try:
            conexion = self.conectar_sin_bd()
            if not conexion:
                return False
            
            cursor = conexion.cursor()
            cursor.execute(f"CREATE DATABASE IF NOT EXISTS {self.database}")
            conexion.commit()
            print(f"✓ Base de datos '{self.database}' creada/verificada")
            
            cursor.close()
            conexion.close()
            return True
            
        except Error as e:
            print(f"✗ Error al crear base de datos: {e}")
            return False
    
    def conectar(self):
        """Establece conexión con la base de datos MySQL"""
        try:
            self.conexion = mysql.connector.connect(
                host=self.host,
                user=self.user,
                password=self.password,
                database=self.database
            )
            self.cursor = self.conexion.cursor()
            print(f"✓ Conectado a MySQL: {self.database}")
            return self.conexion
        except Error as e:
            print(f"✗ Error de conexión: {e}")
            return None
    
    def desconectar(self):
        """Cierra la conexión"""
        if self.conexion and self.conexion.is_connected():
            self.cursor.close()
            self.conexion.close()
            print("✓ Desconectado de MySQL")
    
    def crear_tablas(self):
        """Crea la estructura de tablas relacional"""
        try:
            # TABLA 1: DEPARTMENTS (Departamentos)
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS departments (
                    dept_id INT AUTO_INCREMENT PRIMARY KEY,
                    department VARCHAR(100) UNIQUE NOT NULL
                )
            """)
            
            # TABLA 2: JOB_ROLES (Roles de trabajo)
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS job_roles (
                    role_id INT AUTO_INCREMENT PRIMARY KEY,
                    job_role VARCHAR(100) UNIQUE NOT NULL
                )
            """)
            
            # TABLA 3: EDUCATION_FIELDS (Campos de educación)
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS education_fields (
                    field_id INT AUTO_INCREMENT PRIMARY KEY,
                    education_field VARCHAR(100) UNIQUE NOT NULL
                )
            """)
            
            # TABLA 4: EMPLOYEES (Principal - Empleados)
            self.cursor.execute("""
                CREATE TABLE IF NOT EXISTS employees (
                    emp_id INT AUTO_INCREMENT PRIMARY KEY,
                    employee_number INT UNIQUE NOT NULL,
                    age INT,
                    gender VARCHAR(20),
                    distance_from_home INT,
                    education INT,
                    dept_id INT NOT NULL,
                    role_id INT NOT NULL,
                    field_id INT NOT NULL,
                    environment_satisfaction INT,
                    job_involvement INT,
                    job_level INT,
                    job_satisfaction INT,
                    monthly_income DECIMAL(10, 2),
                    hourly_rate INT,
                    over_time VARCHAR(10),
                    marital_status VARCHAR(20),
                    attrition VARCHAR(10),
                    business_travel VARCHAR(50),
                    years_at_company INT,
                    years_in_current_role INT,
                    performance_rating INT,
                    total_working_years INT,
                    training_times_last_year INT,
                    num_companies_worked INT,
                    percent_salary_hike DECIMAL(5, 2),
                    stock_option_level INT,
                    work_life_balance INT,
                    
                    FOREIGN KEY (dept_id) REFERENCES departments(dept_id),
                    FOREIGN KEY (role_id) REFERENCES job_roles(role_id),
                    FOREIGN KEY (field_id) REFERENCES education_fields(field_id)
                )
            """)
            
            self.conexion.commit()
            print("✓ Tablas creadas correctamente")
            
        except Error as e:
            print(f"✗ Error al crear tablas: {e}")
            self.conexion.rollback()
    
    def insertar_dimensiones(self, df):
        """Inserta valores únicos en las tablas dimensionales"""
        try:
            # Insertar departamentos únicos
            depts = df['department'].unique()
            for dept in depts:
                self.cursor.execute(
                    "INSERT IGNORE INTO departments (department) VALUES (%s)",
                    (dept,)
                )
            self.conexion.commit()
            print(f"✓ {len(depts)} departamentos insertados")
            
            # Insertar roles únicos
            roles = df['job_role'].unique()
            for role in roles:
                self.cursor.execute(
                    "INSERT IGNORE INTO job_roles (job_role) VALUES (%s)",
                    (role,)
                )
            self.conexion.commit()
            print(f"✓ {len(roles)} roles insertados")
            
            # Insertar campos de educación únicos
            fields = df['education_field'].unique()
            for field in fields:
                self.cursor.execute(
                    "INSERT IGNORE INTO education_fields (education_field) VALUES (%s)",
                    (field,)
                )
            self.conexion.commit()
            print(f"✓ {len(fields)} campos de educación insertados")
            
        except Error as e:
            print(f"✗ Error al insertar dimensiones: {e}")
            self.conexion.rollback()
    
    def obtener_id(self, tabla, columna, valor):
        """Obtiene el ID de un valor en una tabla"""
        try:
            if tabla == 'departments':
                id_columna = 'dept_id'
            elif tabla == 'job_roles':
                id_columna = 'role_id'
            elif tabla == 'education_fields':
                id_columna = 'field_id'
            
            self.cursor.execute(
                f"SELECT {id_columna} FROM {tabla} WHERE {columna} = %s",
                (valor,)
            )
            resultado = self.cursor.fetchone()
            return resultado[0] if resultado else None
        except Error as e:
            print(f"✗ Error al obtener ID: {e}")
            return None
    
    def insertar_empleados(self, df):
        """Inserta todos los empleados desde el DataFrame"""
        try:
            contador = 0
            errores = 0
            
            for idx, row in df.iterrows():
                try:
                    dept_id = self.obtener_id('departments', 'department', row['department'])
                    role_id = self.obtener_id('job_roles', 'job_role', row['job_role'])
                    field_id = self.obtener_id('education_fields', 'education_field', row['education_field'])
                    
                    self.cursor.execute("""
                        INSERT INTO employees (
                            employee_number, age, gender, distance_from_home, education,
                            dept_id, role_id, field_id, environment_satisfaction, job_involvement,
                            job_level, job_satisfaction, monthly_income, hourly_rate, over_time,
                            marital_status, attrition, business_travel, years_at_company,
                            years_in_current_role, performance_rating, total_working_years,
                            training_times_last_year, num_companies_worked, percent_salary_hike,
                            stock_option_level, work_life_balance
                        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    """, (
                        int(row['employee_number']),
                        int(row['age']) if pd.notna(row['age']) else None,
                        row['gender'],
                        int(row['distance_from_home']) if pd.notna(row['distance_from_home']) else None,
                        int(row['education']) if pd.notna(row['education']) else None,
                        dept_id,
                        role_id,
                        field_id,
                        int(row['environment_satisfaction']) if pd.notna(row['environment_satisfaction']) else None,
                        int(row['job_involvement']) if pd.notna(row['job_involvement']) else None,
                        int(row['job_level']) if pd.notna(row['job_level']) else None,
                        int(row['job_satisfaction']) if pd.notna(row['job_satisfaction']) else None,
                        float(row['monthly_income']) if pd.notna(row['monthly_income']) else None,
                        int(row['hourly_rate']) if pd.notna(row['hourly_rate']) else None,
                        row['over_time'],
                        row['marital_status'],
                        row['attrition'],
                        row['business_travel'],
                        int(row['years_at_company']) if pd.notna(row['years_at_company']) else None,
                        int(row['years_in_current_role']) if pd.notna(row['years_in_current_role']) else None,
                        int(row['performance_rating']) if pd.notna(row.get('performance_rating')) else None,
                        int(row['total_working_years']) if pd.notna(row.get('total_working_years')) else None,
                        int(row['training_times_last_year']) if pd.notna(row.get('training_times_last_year')) else None,
                        int(row['num_companies_worked']) if pd.notna(row.get('num_companies_worked')) else None,
                        float(row['percent_salary_hike']) if pd.notna(row.get('percent_salary_hike')) else None,
                        int(row['stock_option_level']) if pd.notna(row.get('stock_option_level')) else None,
                        int(row['work_life_balance']) if pd.notna(row.get('work_life_balance')) else None,
                    ))
                    
                    contador += 1
                    
                except Exception as e:
                    errores += 1
                    continue
            
            self.conexion.commit()
            print(f"✓ {contador} empleados insertados ({errores} errores)")
            
        except Error as e:
            print(f"✗ Error al insertar empleados: {e}")
            self.conexion.rollback()
    
    def ejecutar_flujo_completo(self, ruta_csv):
        """Ejecuta todo el flujo de creación e inserción de datos"""
        try:
            print("\n" + "="*80)
            print("CREACIÓN DE BASE DE DATOS HR - ABC CORPORATION")
            print("="*80 + "\n")
            
            # 1. Crear base de datos
            if not self.crear_base_de_datos():
                return
            
            # 2. Conectar
            if not self.conectar():
                return
            
            # 3. Crear tablas
            self.crear_tablas()
            
            # 4. Cargar datos del CSV
            print(f"\nCargando datos desde: {ruta_csv}")
            df = pd.read_csv(ruta_csv)
            print(f"✓ {len(df)} registros cargados")
            
            # 5. Insertar dimensiones
            print("\nInsertando datos dimensionales...")
            self.insertar_dimensiones(df)
            
            # 6. Insertar empleados
            print("\nInsertando datos de empleados...")
            self.insertar_empleados(df)
            
            print("\n✓ PROCESO COMPLETADO EXITOSAMENTE\n")
            
        except Exception as e:
            print(f"✗ Error en el flujo: {e}")
        finally:
            self.desconectar()

In [9]:
# ==============================================================================
# EJECUCIÓN
# ==============================================================================

if __name__ == "__main__":
    
    # Crear instancia de la base de datos con credenciales de MySQL
    bd = BaseDatosHR(
        host='127.0.0.1',
        user='root',
        password=input('password_MySQL'),  # Te pedirá un input de tu contraseña
        database='abc_hr',
        auth_plugin = 'mysql_native_password'
    )
    
    # Ejecutar flujo completo
    bd.ejecutar_flujo_completo("../df_hr_limpio.csv")


CREACIÓN DE BASE DE DATOS HR - ABC CORPORATION

✓ Base de datos 'abc_hr' creada/verificada
✓ Conectado a MySQL: abc_hr
✓ Tablas creadas correctamente

Cargando datos desde: ../df_hr_limpio.csv
✓ 1470 registros cargados

Insertando datos dimensionales...
✓ 3 departamentos insertados
✓ 9 roles insertados
✓ 6 campos de educación insertados

Insertando datos de empleados...
✓ 1470 empleados insertados (0 errores)

✓ PROCESO COMPLETADO EXITOSAMENTE

✓ Desconectado de MySQL
